# Feature Research

Compute features, analyze distributions, and test predictiveness.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Load Data

In [ ]:
from crypto_quant.data.loaders import load_canonical_dataset

data = load_canonical_dataset()
print(f"Loaded data: {data.shape}")

## Compute Features

In [ ]:
from crypto_quant.features.returns import simple_returns
from crypto_quant.features.volatility import realized_volatility
from crypto_quant.features.volume import volume_zscore, taker_buy_ratio

# Compute returns at different horizons
data['return_5m'] = simple_returns(data['close'], periods=5)
data['return_15m'] = simple_returns(data['close'], periods=15)
data['return_30m'] = simple_returns(data['close'], periods=30)
data['return_1h'] = simple_returns(data['close'], periods=60)

# Compute volatility
data['vol_5m'] = realized_volatility(simple_returns(data['close']), window=5, annualized=False)
data['vol_30m'] = realized_volatility(simple_returns(data['close']), window=30, annualized=False)

# Compute volume metrics
data['vol_zscore'] = volume_zscore(data['volume'], window=20)
if 'taker_buy_volume' in data.columns:
    data['taker_buy_ratio'] = taker_buy_ratio(data['taker_buy_volume'], data['volume'])

print("Features computed")
data[['return_5m', 'vol_5m', 'vol_zscore']].head()

## Target Variable: Forward Returns

In [ ]:
# Compute forward returns (for testing predictiveness)
# IMPORTANT: This is future data and should never be used as a feature!
data['forward_return_5m'] = simple_returns(data['close'], periods=-5)  # Negative period = look forward
data['forward_return_15m'] = simple_returns(data['close'], periods=-15)
data['forward_return_30m'] = simple_returns(data['close'], periods=-30)

print("Forward returns computed")

## Feature Distributions

In [ ]:
# Plot feature distributions
features_to_plot = ['return_5m', 'vol_5m', 'vol_zscore']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feature in zip(axes, features_to_plot):
    data[feature].dropna().hist(bins=50, ax=ax)
    ax.set_title(f'Distribution of {feature}')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Feature Correlation with Target

In [ ]:
from crypto_quant.research.correlations import CorrelationAnalyzer

features = data[['return_5m', 'vol_5m', 'vol_zscore']].copy()
target = data['forward_return_5m'].copy()

correlations = CorrelationAnalyzer.feature_target_correlation(features, target)
print("Feature correlations with forward 5m return:")
print(correlations)

# Plot
plt.figure(figsize=(8, 5))
correlations.plot(kind='barh')
plt.title('Feature Correlation with Forward 5m Return')
plt.xlabel('Correlation')
plt.tight_layout()
plt.show()

## Information Coefficient

In [ ]:
from crypto_quant.research.ic import ICCalculator

# Calculate IC for each feature
for feature in features.columns:
    ic = ICCalculator.calculate_ic(
        features[feature].dropna(),
        target.loc[features[feature].dropna().index],
        method='spearman'
    )
    print(f"IC for {feature}: {ic:.6f}")

## Rolling IC

In [ ]:
# Rolling IC analysis
feature_col = 'return_5m'
rolling_ic = ICCalculator.calculate_rolling_ic(
    data[feature_col].dropna(),
    data['forward_return_5m'].loc[data[feature_col].dropna().index],
    window=60,
    method='spearman'
)

plt.figure(figsize=(14, 5))
plt.plot(rolling_ic.index, rolling_ic.values, linewidth=1)
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.title(f'Rolling IC (60-period): {feature_col}')
plt.xlabel('Date')
plt.ylabel('IC')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Mean IC: {rolling_ic.mean():.6f}")
print(f"IC std dev: {rolling_ic.std():.6f}")